# 00 - Colab setup check

Every notebook in this repo starts with the same bootstrap cell (Drive mount, clone or pull,
install). This notebook runs only that cell plus environment checks, so use it once to confirm
the GitHub token, Drive and GPU are working.

Prerequisite: Colab secret `GITHUB_TOKEN_FEDCTI` (key icon, left sidebar, notebook access on).
See `docs/colab_workflow.md`.

In [ ]:
GITHUB_USER, REPO_NAME, BRANCH = "khalerishabh", "fedcti-llm-v3", "main"
REPO_DIR = f"/content/{REPO_NAME}"

import os, subprocess, sys
from google.colab import drive, userdata

drive.mount("/content/drive")
token = userdata.get("GITHUB_TOKEN_FEDCTI")
auth_url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"


def git(*args):
    res = subprocess.run(["git", *args], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError((res.stderr or res.stdout).replace(token, "***"))
    return res.stdout


if os.path.isdir(f"{REPO_DIR}/.git"):
    git("-C", REPO_DIR, "pull", "-q", "--ff-only", auth_url, BRANCH)
else:
    git("clone", "-q", "--branch", BRANCH, auth_url, REPO_DIR)
    git("-C", REPO_DIR, "remote", "set-url", "origin",
        f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
git("-C", REPO_DIR, "config", "user.name", "Rishabh Khale")
git("-C", REPO_DIR, "config", "user.email", "khalerishabh998@gmail.com")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{REPO_DIR}/requirements.txt", "-e", REPO_DIR], check=True)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
print(git("-C", REPO_DIR, "log", "--oneline", "-1"))

In [ ]:
import torch
from fedcti.config import load_config, storage_root
from fedcti.repro import save_metadata

cfg = load_config("exp2/data.yaml")
root = storage_root(cfg)
print("storage root:", root)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())
print(save_metadata(root / "logs" / "setup_smoke.json", cfg))